# Notebook 04 — SA-CCR: Standardised Approach for CCR

This notebook implements and analyses the **SA-CCR** (Standardised Approach for Counterparty Credit Risk)
as defined by the **Basel Committee (BCBS 2014)** and incorporated into **CRR2/CRR3** (EU).

SA-CCR replaced the previous Current Exposure Method (CEM) and Standardised Method (SM).
It is the mandatory fallback for banks not approved for the **IMM** (Internal Model Method).

## Formula

$$EAD = \alpha \times (RC + PFE)$$

Where:
- $\alpha = 1.4$ (regulatory multiplier)
- $RC$ = Replacement Cost (current loss if counterparty defaults today)
- $PFE = \text{multiplier} \times \text{AddOn}_{agg}$ = Potential Future Exposure

## Notebook Contents

1. Single-trade EAD decomposition (IRS)
2. Multi-asset class portfolio analysis
3. Add-on sensitivity by maturity bucket (IR)
4. PFE multiplier dynamics (over-collateralisation)
5. Margined vs unmargined EAD comparison
6. MPOR sensitivity on SA-CCR
7. SA-CCR vs IMM comparison
8. Capital charge table (Basel RWA)

---
**References:** BCBS (2014), EBA (2020) Guidelines on SA-CCR, CRR2 Article 274–279h

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from ccr.sa_ccr import SACCR, SACCRTrade, SUPERVISORY_FACTORS, ALPHA
from ccr.simulation import HullWhite1F, GBM
from ccr.exposure import ExposureProfile, NettingSet
from ccr.collateral import CSAParameters, VMEngine, IMEngine

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports OK — SA-CCR analysis ready')

## 1. SA-CCR Framework — Supervisory Parameters

The supervisory parameters are calibrated by BCBS to reflect the risk of each asset class.

| Asset Class | Supervisory Factor | Correlation ρ |
|---|---|---|
| Interest Rate | 0.50% | 100% intra-bucket |
| FX | 4.00% | N/A (standalone) |
| Credit IG | 0.38% | 50% |
| Credit HY | 1.06% | 50% |
| Equity Large Cap | 32% | 50% |
| Equity Small Cap | 20% | N/A |
| Commodity | 18-40% | 40% |

In [ ]:
# Visualise supervisory factors
sf_labels = list(SUPERVISORY_FACTORS.keys())
sf_values = [v * 100 for v in SUPERVISORY_FACTORS.values()]

color_map = {
    'ir': 'steelblue',
    'fx': 'orange',
    'credit': 'green',
    'equity': 'purple',
    'commodity': 'brown',
}
colors = []
for lbl in sf_labels:
    for prefix, col in color_map.items():
        if lbl.startswith(prefix):
            colors.append(col)
            break

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(sf_labels, sf_values, color=colors, alpha=0.8, edgecolor='white')
ax.set_ylabel('Supervisory Factor (%)')
ax.set_title('SA-CCR Supervisory Factors by Asset Class (BCBS 2014, Annex A)', fontweight='bold')
ax.set_yscale('log')
ax.set_xticklabels(sf_labels, rotation=30, ha='right', fontsize=9)

# Legend
legend_patches = [mpatches.Patch(color=col, label=cls.upper())
                  for cls, col in color_map.items()]
ax.legend(handles=legend_patches, fontsize=9)

for bar, val in zip(bars, sf_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

## 2. Single-Trade EAD Decomposition — IRS

For a payer IRS, the **adjusted notional** includes the **supervisory duration (SD)**:
$$d^{IR}_{adj} = N \times SD, \quad SD = \frac{e^{-0.05S} - e^{-0.05E}}{0.05}$$

where $S$ = start date, $E$ = end date. This captures the interest rate sensitivity
without needing an explicit DV01 calculation.

In [ ]:
# Single payer IRS: EUR 10M notional, 5Y maturity, current MtM = +50k
irs_trade = SACCRTrade(
    trade_id='IRS_PAYER_5Y',
    asset_class='ir',
    notional=10_000_000,
    current_mtm=50_000,
    maturity=5.0,
    is_bought=True,
)

calc = SACCR([irs_trade], is_margined=True, threshold=0.0, mta=0.0)
report_df = calc.report()

print('══ SA-CCR Report: Payer IRS, EUR 10M, 5Y ════════════════════════════════')
print(report_df.to_string(index=False))

# Visualise EAD waterfall
rc      = calc.replacement_cost()
addon   = calc.addon_ir()
mult    = calc.pfe_multiplier()
pfe     = mult * addon
ead     = calc.ead()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Waterfall ───────────────────────────────────────────────────────────
ax = axes[0]
components = ['RC', 'IR AddOn\n(raw)', 'PFE\n(mult×AddOn)', 'RC + PFE', 'EAD\n(1.4×)']
values     = [rc, addon, pfe, rc + pfe, ead]
colors_w   = ['steelblue', 'orange', 'green', 'teal', 'tomato']
bars = ax.bar(components, [v/1e3 for v in values], color=colors_w, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'€{val/1e3:.1f}k', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Amount (EUR thousands)')
ax.set_title('SA-CCR Decomposition: Payer IRS 5Y', fontweight='bold')

# ── Right: Add-on by maturity bucket ─────────────────────────────────────────
ax2 = axes[1]
# Show how adjusted notional changes with maturity (SD effect)
maturities_show = [0.5, 1, 2, 3, 5, 7, 10, 15, 20]
adj_notionals   = []
for mat in maturities_show:
    t_tmp = SACCRTrade('tmp', 'ir', 10_000_000, 0, mat, is_bought=True)
    c_tmp = SACCR([t_tmp], is_margined=False)
    adj_notionals.append(c_tmp.adjusted_notional(t_tmp) / 1e6)

ax2.bar([str(m) for m in maturities_show], adj_notionals, color='steelblue', alpha=0.8)
ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Adjusted Notional (EUR millions)')
ax2.set_title('IR Adjusted Notional vs Maturity\n(Supervisory Duration Effect)', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n  Maturity factor (margined, 10d MPOR): {calc.maturity_factor(irs_trade):.4f}')
print(f'  Supervisory delta (linear, long):    {calc.supervisory_delta(irs_trade):.1f}')
print(f'  Adjusted notional (IR, 5Y):          EUR {calc.adjusted_notional(irs_trade)/1e6:.4f}M')

## 3. Multi-Asset Class Portfolio

A realistic CCR portfolio contains multiple asset classes. SA-CCR computes add-ons
**independently per asset class** and sums them — there is **no cross-asset netting benefit**.

This is one of SA-CCR's simplifications vs the IMM, which can capture cross-asset correlations.

In [ ]:
# Multi-asset portfolio
portfolio = [
    # IRS: payer 5Y
    SACCRTrade('IRS_PAY_5Y',  'ir',     10_000_000,  50_000, maturity=5.0,  is_bought=True),
    # IRS: receiver 10Y (partial offset in IR bucket)
    SACCRTrade('IRS_REC_10Y', 'ir',      5_000_000, -20_000, maturity=10.0, is_bought=False),
    # FX Forward: long EUR/USD
    SACCRTrade('FX_EURUSD',   'fx',      2_000_000,  10_000, maturity=1.0,  is_bought=True),
    # FX Forward: short GBP/USD (hedged somewhat)
    SACCRTrade('FX_GBPUSD',   'fx',      1_500_000, -5_000,  maturity=0.5,  is_bought=False),
    # CDS: IG protection buyer
    SACCRTrade('CDS_IG',      'credit',    500_000,   8_000, maturity=5.0,  is_bought=True,  sub_type='ig'),
    # Equity option: long call on large cap
    SACCRTrade('EQ_CALL',     'equity',    300_000,  15_000, maturity=1.0,  is_bought=True,
               option_type='call', sub_type='large'),
]

calc_port = SACCR(portfolio, is_margined=True)
report_port = calc_port.report()

print('══ Multi-Asset Portfolio SA-CCR Report ══════════════════════════════════')
print(report_port.to_string(index=False))

# Waterfall chart
addons_by_class = {
    'IR':        calc_port.addon_ir(),
    'FX':        calc_port.addon_fx(),
    'Credit':    calc_port.addon_credit(),
    'Equity':    calc_port.addon_equity(),
    'Commodity': calc_port.addon_commodity(),
}
addon_agg = sum(addons_by_class.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Add-on composition ──────────────────────────────────────────────────
ax = axes[0]
labels_ac = [k for k in addons_by_class if addons_by_class[k] > 0]
vals_ac   = [addons_by_class[k]/1e3 for k in labels_ac]
colors_ac = ['steelblue', 'orange', 'green', 'purple', 'brown'][:len(labels_ac)]
wedges, texts, autotexts = ax.pie(
    vals_ac, labels=labels_ac, colors=colors_ac,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75
)
ax.set_title('AddOn Composition by Asset Class', fontweight='bold')

# ── Right: Full EAD breakdown ─────────────────────────────────────────────────
ax2 = axes[1]
rc_p  = calc_port.replacement_cost()
mult_p = calc_port.pfe_multiplier()
pfe_p  = mult_p * addon_agg
ead_p  = calc_port.ead()

# Stacked bar: PFE components
addon_vals = [addons_by_class[k] for k in ['IR', 'FX', 'Credit', 'Equity', 'Commodity']]
addon_cols = ['steelblue', 'orange', 'green', 'purple', 'brown']
bottom = 0
for k, v, col in zip(['IR', 'FX', 'Credit', 'Equity', 'Commodity'], addon_vals, addon_cols):
    ax2.bar(1, v*mult_p/1e3, bottom=bottom/1e3, color=col, alpha=0.8, width=0.4, label=f'PFE {k}')
    bottom += v * mult_p
ax2.bar(0.5, rc_p/1e3, color='tomato', alpha=0.8, width=0.4, label='RC')
ax2.bar(1.5, ead_p/1e3, color='black', alpha=0.7, width=0.4, label='EAD (total)')

ax2.set_xticks([0.5, 1.0, 1.5])
ax2.set_xticklabels(['RC', 'PFE (by class)', 'EAD'])
ax2.set_ylabel('EUR thousands')
ax2.set_title('SA-CCR: RC + PFE → EAD', fontweight='bold')
ax2.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

print(f'\n  AddOn aggregate : EUR {addon_agg/1e3:.1f}k')
print(f'  PFE multiplier  : {mult_p:.4f}')
print(f'  PFE             : EUR {pfe_p/1e3:.1f}k')
print(f'  RC              : EUR {rc_p/1e3:.1f}k')
print(f'  EAD             : EUR {ead_p/1e3:.1f}k')

## 4. Netting Benefit — SA-CCR vs CEM

SA-CCR captures **within-asset-class netting** via the supervisory delta (δ = ±1).
Opposite-direction trades in the same asset class **partially cancel** in the add-on.

However, SA-CCR **does not allow cross-asset netting** — IR and FX add-ons are always summed.
This is conservative relative to the IMM which captures cross-asset correlations.

In [ ]:
# Demonstrate netting benefit within IR asset class
# Scenario A: payer only
# Scenario B: payer + receiver (offsetting in same maturity bucket)
# Scenario C: payer + receiver (different maturity buckets — less offset)

payer_5y   = SACCRTrade('PAY_5Y',  'ir', 10_000_000,  50_000, maturity=5.0,  is_bought=True)
recv_5y    = SACCRTrade('REC_5Y',  'ir', 10_000_000, -50_000, maturity=5.0,  is_bought=False)
recv_10y   = SACCRTrade('REC_10Y', 'ir', 10_000_000, -50_000, maturity=10.0, is_bought=False)
recv_2y    = SACCRTrade('REC_2Y',  'ir', 10_000_000, -50_000, maturity=2.0,  is_bought=False)

scenarios = {
    'Payer 5Y only':               SACCR([payer_5y]),
    'Payer 5Y + Receiver 5Y':      SACCR([payer_5y, recv_5y]),
    'Payer 5Y + Receiver 10Y':     SACCR([payer_5y, recv_10y]),
    'Payer 5Y + Receiver 2Y':      SACCR([payer_5y, recv_2y]),
}

print('── IR Netting Benefit within SA-CCR ─────────────────────────────────────')
print(f'{"Scenario":<35} {"RC":>10} {"IR AddOn":>12} {"EAD":>10}')
print('-' * 70)
results_net = {}
for label, calc_s in scenarios.items():
    rc_s   = calc_s.replacement_cost()
    add_s  = calc_s.addon_ir()
    ead_s  = calc_s.ead()
    results_net[label] = {'RC': rc_s, 'AddOn': add_s, 'EAD': ead_s}
    print(f'{label:<35} {rc_s/1e3:>8.1f}k  {add_s/1e3:>10.1f}k  {ead_s/1e3:>8.1f}k')

fig, ax = plt.subplots(figsize=(10, 4))
labels_n = list(results_net.keys())
eads_n   = [results_net[l]['EAD']/1e3 for l in labels_n]
adds_n   = [results_net[l]['AddOn']/1e3 for l in labels_n]
x_n = np.arange(len(labels_n))

ax.bar(x_n - 0.2, eads_n, 0.35, color='tomato',    alpha=0.8, label='EAD')
ax.bar(x_n + 0.2, adds_n, 0.35, color='steelblue', alpha=0.7, label='IR AddOn')
ax.set_xticks(x_n)
ax.set_xticklabels([l.replace(' + ', '\n+ ') for l in labels_n], fontsize=8)
ax.set_ylabel('EUR thousands')
ax.set_title('SA-CCR Netting: IR Same vs Cross Buckets', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. PFE Multiplier — Over-Collateralisation Effect

The **PFE multiplier** rewards over-collateralisation by reducing the PFE component:

$$\text{multiplier} = \min\left(1,\; 0.05 + 0.95 \cdot \exp\left(\frac{V - C}{2 \times 0.95 \times \text{AddOn}_{agg}}\right)\right)$$

- When $V - C = 0$ (collateral exactly covers MtM): multiplier = $0.05 + 0.95 = 1.0$
- When $C > V$ (over-collateralised): multiplier < 1.0 (floored at 0.05)
- When $C < V$ (under-collateralised): multiplier > 1.0 → capped at 1.0

In [ ]:
# Show PFE multiplier as a function of VM held (collateral)
irs_for_mult = SACCRTrade('IRS', 'ir', 10_000_000, current_mtm=100_000, maturity=5.0, is_bought=True)

# Vary VM held from -200k (owed by us) to +400k (received)
vm_range = np.linspace(-200_000, 400_000, 200)
multipliers = []
pfe_vals    = []
ead_vals    = []

for vm in vm_range:
    c = SACCR([irs_for_mult], is_margined=True, vm_held=max(vm, 0))
    multipliers.append(c.pfe_multiplier())
    addon_agg = c.addon_ir()
    pfe_vals.append(c.pfe_multiplier() * addon_agg)
    ead_vals.append(c.ead())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(vm_range/1e3, multipliers, color='steelblue', lw=2.5)
ax.axvline(0, color='grey', ls=':', lw=1)
ax.axvline(100, color='orange', ls='--', lw=1.5, label='V-C=0 (VM=MtM=100k)')
ax.axhline(1.0, color='black', ls=':', lw=1, label='Cap at 1.0')
ax.axhline(0.05, color='tomato', ls=':', lw=1, label='Floor at 0.05')
ax.fill_between(vm_range/1e3, multipliers, 1.0, where=np.array(multipliers) < 1.0,
                alpha=0.2, color='green', label='Over-collat. benefit')
ax.set_xlabel('VM held (EUR thousands)')
ax.set_ylabel('PFE Multiplier')
ax.set_title('PFE Multiplier vs Collateral Held', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)

ax2 = axes[1]
ax2.plot(vm_range/1e3, [v/1e3 for v in ead_vals], color='tomato', lw=2.5, label='EAD')
ax2.plot(vm_range/1e3, [v/1e3 for v in pfe_vals], color='orange', lw=2.0, ls='--', label='PFE')
ax2.axvline(100, color='grey', ls=':', lw=1, label='VM = MtM (collat. neutral)')
ax2.set_xlabel('VM held (EUR thousands)')
ax2.set_ylabel('Amount (EUR thousands)')
ax2.set_title('EAD and PFE vs Collateral Held', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Key values
print('── PFE Multiplier Key Values ──────────────────────────────────────────────')
for vm_val in [-200_000, 0, 50_000, 100_000, 200_000, 400_000]:
    c = SACCR([irs_for_mult], is_margined=True, vm_held=max(vm_val, 0))
    print(f'  VM={vm_val/1e3:>7.0f}k → multiplier={c.pfe_multiplier():.4f}, EAD={c.ead()/1e3:.1f}k')

## 6. Margined vs Unmargined EAD

SA-CCR distinguishes two regimes:

**Unmargined:**
- RC = max(V - C, 0)
- MF = √(min(M, 1y) / 1y)  — scales with full remaining maturity

**Margined:**
- RC = max(V - C, TH + MTA - NICA, 0)  — includes threshold floor
- MF = √(MPOR / 1y) = √(10/252) ≈ 0.199  — only MPOR horizon

The margined EAD is typically **much lower** because the maturity factor is bounded by the MPOR
rather than the full trade maturity.

In [ ]:
maturities_cmp = [0.5, 1, 2, 3, 5, 7, 10, 15, 20]

ead_margined   = []
ead_unmargined = []
mf_margined    = []
mf_unmargined  = []

for mat in maturities_cmp:
    t = SACCRTrade('IRS', 'ir', 10_000_000, 50_000, maturity=mat, is_bought=True)
    c_m  = SACCR([t], is_margined=True)
    c_um = SACCR([t], is_margined=False)
    ead_margined.append(c_m.ead())
    ead_unmargined.append(c_um.ead())
    mf_margined.append(c_m.maturity_factor(t))
    mf_unmargined.append(c_um.maturity_factor(t))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(maturities_cmp, [v/1e3 for v in ead_unmargined], color='tomato',    lw=2.5, marker='o',
        label='Unmargined (MF = √(min(M,1y)/1y))')
ax.plot(maturities_cmp, [v/1e3 for v in ead_margined],   color='steelblue', lw=2.5, marker='s',
        label='Margined (MF = √(10d/1y) fixed)')
ax.set_xlabel('Trade Maturity (years)')
ax.set_ylabel('EAD (EUR thousands)')
ax.set_title('EAD by Maturity: Margined vs Unmargined', fontweight='bold')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.plot(maturities_cmp, mf_unmargined, color='tomato',    lw=2.5, marker='o', label='Unmargined MF')
ax2.axhline(mf_margined[0], color='steelblue', lw=2.5, ls='--',
            label=f'Margined MF = {mf_margined[0]:.4f} (const.)')
ax2.set_xlabel('Trade Maturity (years)')
ax2.set_ylabel('Maturity Factor')
ax2.set_title('Maturity Factor: Capped at √(1y) for Unmargined,\nFixed at √(MPOR) for Margined', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('── EAD: Margined vs Unmargined ───────────────────────────────────────────')
print(f'{"Maturity":>8} {"EAD Margined":>15} {"EAD Unmargined":>17} {"Ratio":>8}')
print('-' * 52)
for mat, em, eu in zip(maturities_cmp, ead_margined, ead_unmargined):
    print(f'{mat:>6.1f}y   {em/1e3:>11.1f}k    {eu/1e3:>13.1f}k   {eu/em:>6.2f}×')

## 7. IR Maturity Bucket Analysis

SA-CCR groups IR trades into **three maturity buckets** for aggregation:
- **Bucket 1:** 0–1 year  
- **Bucket 2:** 1–5 years  
- **Bucket 3:** 5+ years  

Within the same bucket and currency: full netting (ρ = 100%).
Cross-bucket, same currency: aggregated as simple sum of absolute values (ρ = 0%).

This means a **payer 5Y + receiver 2Y** netting set has less netting benefit than
a **payer 5Y + receiver 4Y** (both in bucket 2).

In [ ]:
# Demonstrate bucket aggregation: payer 5Y with various receiver maturities
receiver_maturities = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
payer_base = SACCRTrade('PAY_5Y', 'ir', 10_000_000, 50_000, maturity=5.0, is_bought=True)

addon_net_list  = []
addon_gross_list = []
bucket_labels   = []

for mat in receiver_maturities:
    recv = SACCRTrade(f'REC_{mat}Y', 'ir', 10_000_000, -50_000, maturity=mat, is_bought=False)
    calc_net  = SACCR([payer_base, recv], is_margined=True)
    calc_gross = SACCR([payer_base], is_margined=True)  # payer alone
    
    addon_net_list.append(calc_net.addon_ir())
    addon_gross_list.append(calc_gross.addon_ir())
    
    # Determine receiver bucket
    b = 1 if mat <= 1 else (2 if mat <= 5 else 3)
    bucket_labels.append(f'{mat}Y\n(Bkt {b})')

fig, ax = plt.subplots(figsize=(12, 4))
x_b = np.arange(len(receiver_maturities))
ax.bar(x_b - 0.2, [v/1e3 for v in addon_gross_list], 0.35, color='grey',      alpha=0.6, label='Payer 5Y alone (Bkt 2)')
ax.bar(x_b + 0.2, [v/1e3 for v in addon_net_list],   0.35, color='steelblue', alpha=0.8, label='Payer 5Y + Receiver (netting)')

# Shade bucket regions
ax.axvspan(-0.5, 1.5, alpha=0.05, color='red',    label='Bucket 1 receivers (0–1y)')
ax.axvspan(1.5, 7.5,  alpha=0.05, color='green',  label='Bucket 2 receivers (1–5y)')
ax.axvspan(7.5, 9.5,  alpha=0.05, color='blue',   label='Bucket 3 receivers (5y+)')

ax.set_xticks(x_b)
ax.set_xticklabels(bucket_labels, fontsize=8)
ax.set_ylabel('IR AddOn (EUR thousands)')
ax.set_title('IR AddOn: Payer 5Y + Receiver at Different Maturities\n(Netting benefit within same bucket)', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print('─ Best netting: receiver in same bucket as payer (Bucket 2: 1–5y)')
best_idx = np.argmin(addon_net_list)
print(f'  Lowest AddOn: Receiver {receiver_maturities[best_idx]}Y → AddOn={addon_net_list[best_idx]/1e3:.2f}k')

## 8. SA-CCR vs IMM — EAD Comparison

The SA-CCR and IMM (Internal Model Method) represent two different philosophies:

| Feature | SA-CCR | IMM |
|---|---|---|
| Basis | Supervisory formula | Monte Carlo simulation |
| Netting | Within-class only | Full portfolio |
| Collateral | Simplified multiplier | Path-by-path VM/IM |
| Capital metric | EAD = 1.4×(RC+PFE) | EAD = 1.4×EEPE |
| Regulatory approval | Not needed | Required |

Below we compare EAD estimates for the same payer IRS under both approaches.

In [ ]:
# IMM estimate via Monte Carlo
N_PATHS, N_STEPS, HORIZON = 3_000, 60, 5.0
NOTIONAL, FIXED_RATE = 10_000_000, 0.03

hw = HullWhite1F(r0=0.03, a=0.05, sigma=0.010)
sim = hw.simulate(n_paths=N_PATHS, n_steps=N_STEPS, horizon=HORIZON, seed=42)

time_grid  = sim.time_grid[1:]
rate_paths = sim.paths[:, 1:]
T_rem      = HORIZON - time_grid
annuity    = (1.0 - np.exp(-rate_paths * T_rem)) / np.maximum(rate_paths, 1e-6)
mtm_matrix = NOTIONAL * (rate_paths - FIXED_RATE) * annuity

# Gross IMM
prof_gross = ExposureProfile(mtm_matrix, time_grid)
ead_imm_gross = prof_gross.ead_imm()

# VM-only IMM (10d MPOR)
csa = CSAParameters.zero_threshold_bilateral(mpor_days=10)
vm_engine = VMEngine(csa)
exp_vm = vm_engine.collateral_adjusted_exposure(mtm_matrix, time_grid)
prof_vm = ExposureProfile(exp_vm, time_grid)
ead_imm_vm = prof_vm.ead_imm()

# SA-CCR estimate
irs_sacr = SACCRTrade('IRS_PAY_5Y', 'ir', NOTIONAL, current_mtm=50_000, maturity=5.0, is_bought=True)
calc_sacr_m  = SACCR([irs_sacr], is_margined=True)
calc_sacr_um = SACCR([irs_sacr], is_margined=False)
ead_sacr_m   = calc_sacr_m.ead()
ead_sacr_um  = calc_sacr_um.ead()

# Comparison table
print('══ SA-CCR vs IMM EAD Comparison ═════════════════════════════════════════')
print(f'  IMM (gross, no collateral)    : EUR {ead_imm_gross/1e6:.4f}M')
print(f'  IMM (VM-only, 10d MPOR)       : EUR {ead_imm_vm/1e6:.4f}M')
print(f'  SA-CCR (margined, 10d MPOR)   : EUR {ead_sacr_m/1e6:.4f}M')
print(f'  SA-CCR (unmargined)           : EUR {ead_sacr_um/1e6:.4f}M')
print(f'')
print(f'  Ratio SA-CCR margined / IMM VM : {ead_sacr_m / ead_imm_vm:.2f}×')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
labels_cmp = ['IMM\n(no collateral)', 'IMM\n(VM only)', 'SA-CCR\n(margined)', 'SA-CCR\n(unmargined)']
eads_cmp   = [ead_imm_gross, ead_imm_vm, ead_sacr_m, ead_sacr_um]
colors_cmp = ['grey', 'steelblue', 'tomato', 'orange']
bars = ax.bar(labels_cmp, [v/1e6 for v in eads_cmp], color=colors_cmp, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, eads_cmp):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'€{val/1e6:.4f}M', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('EAD (EUR millions)')
ax.set_title('EAD Comparison: SA-CCR vs IMM (Payer IRS EUR 10M, 5Y)', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Capital Charge Table — RWA and Minimum Capital

Under the Basel III framework, the capital requirement for CCR is:

$$K = EAD \times RW \times 8\%$$

Where the risk weight (RW) depends on the counterparty's credit quality:
- BBB (standard bilateral OTC): RW = 20–50%
- Qualified CCP (cleared): RW = 2%
- Financial institutions (bank): RW = 25–100%

In [ ]:
# Capital charge analysis across a realistic trading book
trading_book = [
    # (label, trades, is_margined, risk_weight, description)
    ('Corp A (BBB, bilateral)',
     [SACCRTrade('IRS_5Y', 'ir', 10e6, 80_000, maturity=5.0, is_bought=True)],
     True, 0.50, 'Standard corporate'),
    
    ('Corp B (A-, bilateral)',
     [SACCRTrade('IRS_10Y', 'ir', 5e6, 40_000, maturity=10.0, is_bought=True),
      SACCRTrade('FX_1Y',   'fx', 2e6, 10_000, maturity=1.0,  is_bought=True)],
     True, 0.20, 'Investment grade'),
    
    ('CCP (cleared)',
     [SACCRTrade('IRS_5Y_CCP', 'ir', 10e6, 80_000, maturity=5.0, is_bought=True)],
     True, 0.02, 'Qualified CCP (LCH, CME)'),
    
    ('Hedge Fund (speculative)',
     [SACCRTrade('EQ_OPT', 'equity', 1e6, 25_000, maturity=2.0, is_bought=True,
                 option_type='call', sub_type='small')],
     False, 1.00, 'Unrated / high risk'),
]

CAPITAL_RATIO = 0.08  # Basel III minimum
rows_cap = []
for label, trades, margined, rw, desc in trading_book:
    calc = SACCR(trades, is_margined=margined)
    ead = calc.ead()
    rwa = ead * rw
    capital = rwa * CAPITAL_RATIO
    rows_cap.append({
        'Counterparty': label,
        'EAD (€k)':     f'{ead/1e3:,.1f}',
        'Risk Weight':  f'{rw*100:.0f}%',
        'RWA (€k)':     f'{rwa/1e3:,.1f}',
        'Capital (€k)': f'{capital/1e3:,.2f}',
        'Description':  desc,
    })

df_cap = pd.DataFrame(rows_cap)
print('══ Capital Charge Table ════════════════════════════════════════════════')
print(df_cap.to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cp_labels = [r['Counterparty'].split(' (')[0] + '\n' + r['Counterparty'].split(' (')[1].rstrip(')')
             if ' (' in r['Counterparty'] else r['Counterparty'] for r in rows_cap]
eads_cap  = [float(r['EAD (€k)'].replace(',','')) for r in rows_cap]
rwas_cap  = [float(r['RWA (€k)'].replace(',','')) for r in rows_cap]
caps_cap  = [float(r['Capital (€k)'].replace(',','')) for r in rows_cap]

x_c = np.arange(len(rows_cap))
ax = axes[0]
ax.bar(x_c - 0.25, eads_cap,  0.22, color='steelblue', alpha=0.8, label='EAD')
ax.bar(x_c,        rwas_cap,  0.22, color='orange',    alpha=0.8, label='RWA')
ax.bar(x_c + 0.25, caps_cap,  0.22, color='tomato',    alpha=0.9, label='Capital (8%)')
ax.set_xticks(x_c)
ax.set_xticklabels(cp_labels, fontsize=7)
ax.set_ylabel('EUR thousands')
ax.set_title('SA-CCR Capital by Counterparty', fontweight='bold')
ax.legend(fontsize=9)

# Capital intensity (capital / notional)
ax2 = axes[1]
notionals = [10e6, 7e6, 10e6, 1e6]  # approximate notionals
cap_intensity = [c / n * 100 for c, n in zip([r*1000 for r in caps_cap], notionals)]
bar_cols = ['steelblue', 'green', 'orange', 'tomato']
ax2.bar(cp_labels, cap_intensity, color=bar_cols, alpha=0.8, edgecolor='white')
ax2.set_ylabel('Capital / Notional (%)')
ax2.set_title('Capital Intensity (Capital / Notional)', fontweight='bold')
ax2.set_xticklabels(cp_labels, fontsize=7)

plt.tight_layout()
plt.show()

## 10. SA-CCR Summary — Regulatory Scorecard

| Component | Formula | Key Insight |
|---|---|---|
| **α multiplier** | 1.4 | Conservative buffer for model risk |
| **RC** | max(V−C, TH+MTA−NICA, 0) | Floor = threshold (even if over-collat.) |
| **MF (margined)** | √(MPOR/1y) | Fixed at MPOR regardless of maturity |
| **MF (unmargined)** | √(min(M,1y)/1y) | Capped at 1y — saturates for long trades |
| **Supervisory delta** | ±Φ(d₁) for options | Captures optionality direction |
| **IR adjusted notional** | N×SD | SD compresses notional by duration |
| **PFE multiplier** | min(1, 0.05+0.95exp()) | Floored at 5% — never fully zero |
| **No cross-class netting** | AddOn = ΣAddOn_k | Conservative vs IMM |
| **Clearing benefit** | MPOR 5d vs 10d | √(5/10) ≈ 0.71× factor on add-ons |
| **Dispute penalty** | 2×MPOR | CRR3 Art. 285(4) |

---

**Calibration context:** SA-CCR supervisory factors are calibrated to match IMM EEPE for typical vanilla
portfolios at the 1-year capital horizon. For complex portfolios with strong netting, IMM is typically
20–40% lower than SA-CCR.